# Sentiment Analysis Using Bag of Words and KNearest Neighbors (KNN)

## Project Overview

Natural Language Processing (NLP) enables machines to understand, analyze, and extract meaningful information from human language. One of the most fundamental NLP tasks is **Sentiment Analysis**, which aims to determine whether a piece of text expresses a positive or negative opinion.

In this project, I build a complete sentiment analysis pipeline using classical NLP techniques and Machine Learning algorithms. The workflow covers the entire process from raw text preprocessing to feature extraction, model training, evaluation, and experiment documentation.

The primary objective of this project is not only to achieve high predictive performance but also to systematically investigate how different preprocessing techniques, feature engineering strategies, and machine learning models affect sentiment classification performance.

---

## Dataset

Dataset Link: **[IMBD Movie Reviews](https://www.kaggle.com/datasets/mwallerphunware/imbd-movie-reviews-for-binary-sentiment-analysis)**

The dataset consists of movie reviews labeled with their corresponding sentiment:

* Positive
* Negative

The reviews contain real-world natural language, making them suitable for evaluating text preprocessing techniques and machine learning models.

---

## Project Pipeline

The following stages are implemented throughout this project:

### 1. Text Preprocessing

* Contraction expansion
* Lowercasing
* Text cleaning using regular expressions
* Stopword removal with preserved negations
* Lemmatization
* Corpus construction

### 2. Feature Extraction

* Bag of Words (CountVectorizer)
* N-gram generation
* Vocabulary analysis
* Feature selection using `max_features`
* Rare-word filtering using `min_df`

### 3. Model Training

Different machine learning algorithms are evaluated and compared, including:

* Gaussian Naive Bayes
* Multinomial Naive Bayes
* Bernoulli Naive Bayes
* Logistic Regression
* Support Vector Machines (SVM)
* Decision Trees
* Random Forests

### 4. Model Evaluation

Performance is assessed using multiple metrics:

* Accuracy
* Precision
* Recall
* F1-Score
* ROC-AUC Score
* Confusion Matrix
* Cross-Validation Mean Accuracy
* Cross-Validation Standard Deviation

---

## Experimental Approach

Rather than training a single model, this notebook follows an experimentation-driven methodology.

For each model, multiple configurations are tested, including different values for:

* `max_features`
* `min_df`
* N-gram ranges
* Preprocessing strategies

The goal is to identify the most effective configuration while understanding the trade-offs between model complexity, computational cost, and predictive performance.

---

## Key Learning Objectives

Through this project, I aim to:

* Develop a deeper understanding of NLP preprocessing techniques.
* Compare the behavior of different machine learning algorithms on text data.
* Analyze how feature engineering impacts classification performance.
* Build reproducible NLP pipelines suitable for real-world applications.
* Establish strong baselines before moving toward advanced embedding and transformer-based approaches.

---

**Author:** Hazem Mohamed

**Role:** AI Engineer | Machine Learning Engineer | NLP Engineer

**Repository:** [NLP Experimentation Lab](https://github.com/Hazem1695/NLP-Experimentation-Lab)


# **Importing the Libraries**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## Data Cleaning Check Template
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

## **Load dataset**
Apply Data Cleaning Check Template

In [ ]:
dataset = pd.read_csv('MovieReviewTrainingDatabase.csv')
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  25000 non-null  object
 1   review     25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB
None

SHAPE OF DATA
Rows: 25000, Columns: 2

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 96

DATA TYPES
sentiment    object
review       object
dtype: object

NUMERICAL SUMMARY
       sentiment                                             review
count      25000                                              25000
unique         2                                              24904
top     Positive  You do realize that you've been watching the E...
freq       12500                                                  3

CATEGORICAL SUMMARY
       sentiment                                             review
count      25000                 

## Duplicate Data Detection

In [4]:
duplicates = dataset[dataset.duplicated(subset=['review'], keep=False)]
duplicates.sort_values('review')

,sentiment,review
21186,Negative,"Back in his youth, the old man had wanted to..."
21877,Negative,"Back in his youth, the old man had wanted to..."
14734,Negative,'Dead Letter Office' is a low-budget film abou...
5519,Negative,'Dead Letter Office' is a low-budget film abou...
7011,Positive,".......Playing Kaddiddlehopper, Col San Fernan..."
...,...,...
2685,Negative,"in this movie, joe pesci slams dunks a basketb..."
22244,Positive,it's amazing that so many people that i know h...
14767,Positive,it's amazing that so many people that i know h...
12462,Negative,this movie begins with an ordinary funeral... ...


## Quantifying Duplicate Review Frequencies

In [5]:
review_counts = dataset['review'].value_counts()
print("Reviews appearing more than once:")
print((review_counts > 1).sum())
print("\nMaximum repetitions:")
print(review_counts.max())

Reviews appearing more than once:
92

Maximum repetitions:
3


## Removing Duplicate Reviews & Resetting Index
> **Note:** This cell drops the repeated rows we identified in the previous steps and cleanly resets the row indices for model training

In [6]:
print("Before:", len(dataset))
dataset = dataset.drop_duplicates()
print("After:", len(dataset))
dataset = dataset.reset_index(drop=True)

Before: 25000
After: 24904


## Library Installation
> **Note:** The `contractions` library is required to automatically expand shortcuts like *don't* to *do not* and *I'm* to *I am* during preprocessing.

In [7]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 5.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00


## **Cleaning the texts**

In [8]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
import re
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

base_stopwords = set(stopwords.words('english'))
negation_words = {'not', 'no', 'never'}
all_stopwords = base_stopwords - negation_words

wnl = WordNetLemmatizer()

corpus = []

for i in range(0, len(dataset)):
    review = dataset['review'][i]
    # Fix contractions
    review = contractions.fix(review)
    # Lowercase
    review = review.lower()
    
    review = re.sub(r'[^a-zA-Z\s]', ' ', review)
    # Split
    words = review.split()
    # Chained Lemmatization (Handles both Verbs 'v' and Nouns 'n')
    review = [wnl.lemmatize(wnl.lemmatize(word, pos='v'), pos='n') for word in words if word not in all_stopwords]
    review = ' '.join(review) 
    corpus.append(review)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


## Preprocessing Verification
> **Note:** Pulling the first two rows directly as a memory array to confirm that our lowercasing, stopword stripping, and lemmatization pipeline worked correctly before feeding it into the vectorizer.

In [9]:
# Pull the data directly as a fast memory array
raw_samples = dataset['review'].head(2).values

for i in range(2):
    print(f"=== REVIEW #{i+1} ===")
    print(f"RAW:     {raw_samples[i]}\n") 
    print(f"CLEANED: {corpus[i]}")
    print("-" * 50)

=== REVIEW #1 ===
RAW:     With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.  Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.  The actual feature film bit when it final

# **Creating the Bag of Word model**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# Add ngram_range=(1, 2) so it automatically catches phrases like "not good" or "no clue"
cv = CountVectorizer(max_features=10000, min_df=2, ngram_range=(1, 2))
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:, 0].values

In [11]:
len(X[0])

10000

# **Encoding Categorical data Using Label Encoding**

In [12]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [13]:
print(y)

[1 1 0 ... 0 0 1]


# Class Balance Check
> **Note:** Using NumPy to verify if our dataset is perfectly balanced between positive and negative reviews before splitting it into training and testing sets.

In [14]:
# This returns the unique classes and how many times they appear
classes, counts = np.unique(y, return_counts=True)
for c, count in zip(classes, counts):
    print(f"Class {c} contains {count}")

Class 0 contains 12432
Class 1 contains 12472


# **Splitting the dataset into the Training set and Test set**

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

# **Training the KNearest Neighbors (KNN) model on the Training set**

In [16]:
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors=11, metric='cosine', p=2)
classifier.fit(X_train, y_train)

KNeighborsClassifier(metric='cosine', n_neighbors=11)

# **Predicting the Test set results**

In [17]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

[[1 1]
 [1 1]
 [1 0]
 ...
 [0 0]
 [0 0]
 [0 0]]


# **Evaluating the Model Performance**

In [18]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_pred))


accuracies = cross_val_score(estimator=classifier, X=X_train, y=y_train, cv=3)

print("\nMean Accuracy:")
print(accuracies.mean())

print("\nStandard Deviation:")
print(accuracies.std())

Confusion Matrix:
[[2065  455]
 [ 775 1686]]

Accuracy Score:
0.7530616342099979

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.82      0.77      2520
           1       0.79      0.69      0.73      2461

    accuracy                           0.75      4981
   macro avg       0.76      0.75      0.75      4981
weighted avg       0.76      0.75      0.75      4981


ROC-AUC Score:
0.7522659036525351

Mean Accuracy:
0.7346785122722482

Standard Deviation:
0.003928563354785831


# K-Nearest Neighbors (KNN) Performance Analysis for Text Classification

# 1. Objective

The objective of this experiment was to investigate the effectiveness of the **K-Nearest Neighbors (KNN)** algorithm for binary text classification using a **Bag-of-Words (BoW)** representation generated by **CountVectorizer**.

Unlike probabilistic and linear classifiers, KNN is an instance-based learning algorithm that classifies new documents according to the labels of their nearest training samples. Since text data is represented as sparse and high-dimensional vectors, this experiment also evaluates how different distance metrics and neighborhood sizes influence KNN performance.

The study aims to answer the following research questions:

* How well does KNN perform on sparse textual data?
* What is the impact of increasing the number of neighbors (K)?
* Which distance metric is more suitable for text classification?
* Does removing infrequent words improve model robustness?
* How stable is KNN across different cross-validation folds?

---

# 2. Experimental Setup

## Dataset

* Original dataset cleaned before training.
* Duplicate samples removed.
* Final testing dataset: **4,981 documents**.

---

## Feature Extraction

The textual documents were transformed into numerical vectors using **CountVectorizer**.

The following feature engineering techniques were applied:

| Configuration | Parameters                                      |
| ------------- | ----------------------------------------------- |
| C1            | max_features=5000, ngram_range=(1,2)            |
| C2            | max_features=10000, ngram_range=(1,2)           |
| C3            | max_features=10000, min_df=2, ngram_range=(1,2) |

Feature engineering included:

* Vocabulary size limitation
* Bigram generation
* Rare-word filtering using `min_df=2`

These preprocessing steps reduce dimensionality while preserving informative contextual features.

---

# 3. KNN Configuration

The experiments evaluated multiple values of **K** together with different distance metrics.

```python
KNeighborsClassifier(
    n_neighbors=K,
    metric='minkowski',
    p=2
)
```

Additional experiments replaced Euclidean distance with Cosine similarity:

```python
KNeighborsClassifier(
    n_neighbors=K,
    metric='cosine'
)
```

Distance weighting was also investigated:

```python
weights='distance'
```

---

# 4. Experimental Results

| Configuration                    | Accuracy   | ROC-AUC    | CV Mean    | CV Std     |
| -------------------------------- | ---------- | ---------- | ---------- | ---------- |
| K=3, Euclidean                   | 62.64%     | 0.6251     | 61.57%     | 0.0148     |
| K=5, Euclidean                   | 64.00%     | 0.6386     | 62.46%     | 0.0119     |
| K=5, Euclidean (10K Features)    | 63.96%     | 0.6378     | 61.97%     | 0.0167     |
| K=7, Euclidean                   | 64.30%     | 0.6413     | 62.72%     | 0.0167     |
| K=7, Cosine                      | 74.74%     | 0.7469     | 72.89%     | 0.0052     |
| K=7, Cosine + Distance Weighting | 74.74%     | 0.7469     | 72.91%     | 0.0050     |
| K=9, Cosine                      | 75.03%     | 0.7496     | 73.24%     | 0.0038     |
| **K=11, Cosine**                 | **75.31%** | **0.7523** | **73.47%** | **0.0039** |

---

# 5. Performance Analysis

## Effect of the Number of Neighbors (K)

The number of neighbors had a noticeable impact on model stability and predictive performance.

With **K = 3**, the classifier relied on very few neighboring samples, making predictions highly sensitive to noise and outliers. This resulted in the lowest overall accuracy.

Increasing the neighborhood size to **K = 5**, **K = 7**, **K = 9**, and finally **K = 11** produced progressively smoother decision boundaries, reducing prediction variance and improving classification accuracy.

This behavior is expected because larger values of **K** reduce the influence of noisy observations while allowing predictions to reflect broader local patterns within the data.

---

## Effect of Distance Metric

One of the most significant findings of this study is the dramatic improvement obtained by replacing **Euclidean (Minkowski)** distance with **Cosine distance**.

### Euclidean Distance

Best Accuracy:

**64.30%**

### Cosine Distance

Best Accuracy:

**75.31%**

This represents an improvement of approximately **11 percentage points**, highlighting the importance of selecting an appropriate similarity measure for sparse text representations.

Unlike Euclidean distance, which is influenced by document length and vector magnitude, Cosine distance measures the angular similarity between document vectors. Since Bag-of-Words vectors are extremely sparse and vary considerably in magnitude, cosine similarity provides a far more meaningful measure of document similarity.

This explains the substantial increase in classification performance.

---

## Effect of Distance Weighting

Applying

```python
weights='distance'
```

produced almost identical performance compared to uniform weighting.

Accuracy and ROC-AUC remained virtually unchanged.

This suggests that the nearest neighbors already exhibited similar distances, meaning that assigning larger weights to closer neighbors did not provide additional discriminative information.

---

## Effect of Rare Word Removal

Introducing

```python
min_df = 2
```

removed extremely infrequent terms from the vocabulary.

Although the improvement was relatively small, it produced:

* slightly higher cross-validation accuracy
* lower standard deviation
* improved robustness

Removing noisy features allowed the model to focus on more representative textual patterns.

---

# 6. Precision and Recall Analysis

For the best-performing configuration:

Class 0

* Precision = **0.73**
* Recall = **0.82**

Class 1

* Precision = **0.79**
* Recall = **0.69**

The classifier demonstrated stronger recall for Class 0 than for Class 1.

Although precision remained relatively balanced, the lower recall for the positive class indicates that KNN still misses a considerable number of positive documents, resulting in a larger number of false negatives.

---

# 7. ROC-AUC Analysis

The ROC-AUC score steadily increased as the model configuration improved.

| Configuration   | ROC-AUC    |
| --------------- | ---------- |
| K=3 Euclidean   | 0.6251     |
| K=5 Euclidean   | 0.6386     |
| K=7 Euclidean   | 0.6413     |
| K=7 Cosine      | 0.7469     |
| K=9 Cosine      | 0.7496     |
| **K=11 Cosine** | **0.7523** |

The ROC-AUC trend closely mirrors the accuracy results, confirming that cosine distance consistently improves the model's ability to distinguish between the two classes.

---

# 8. Cross-Validation Analysis

Cross-validation provides insight into the model's ability to generalize beyond the training data.

The best-performing configuration achieved:

* Mean Accuracy = **73.47%**
* Standard Deviation = **0.0039**

The low standard deviation indicates that the optimized KNN model is relatively stable across different data splits.

However, its average performance remains significantly below that of Logistic Regression and Naive Bayes.

---

# 9. Best Model Configuration

The strongest KNN model obtained during the experiments was:

```python
CountVectorizer(
    max_features=10000,
    min_df=2,
    ngram_range=(1,2)
)

KNeighborsClassifier(
    n_neighbors=11,
    metric='cosine'
)
```

Performance:

* Accuracy = **75.31%**
* Precision = **0.76**
* Recall = **0.75**
* F1-score = **0.75**
* ROC-AUC = **0.7523**
* Cross-Validation Accuracy = **73.47%**
* Cross-Validation Standard Deviation = **0.0039**

---

# 10. Comparison with Previously Evaluated Models

| Model                   | Best Accuracy | ROC-AUC    |
| ----------------------- | ------------- | ---------- |
| Gaussian Naive Bayes    | 81.70%        | 0.8156     |
| Multinomial Naive Bayes | 87.56%        | 0.8755     |
| Bernoulli Naive Bayes   | 87.66%        | 0.8767     |
| Logistic Regression     | **88.92%**    | **0.8894** |
| **K-Nearest Neighbors** | **75.31%**    | **0.7523** |

Among all classical machine learning algorithms evaluated, KNN produced the lowest overall performance.

While cosine similarity substantially improved the classifier, it remained significantly behind Logistic Regression and the Naive Bayes variants.

---

# 11. Discussion

The experimental results clearly demonstrate the limitations of KNN for high-dimensional text classification.

Unlike Logistic Regression, which learns a global linear decision boundary, or Naive Bayes, which estimates probabilistic relationships between words and classes, KNN relies entirely on local distance calculations. In sparse Bag-of-Words spaces, most document vectors are nearly orthogonal, making Euclidean distance less informative and degrading neighborhood quality.

Switching to cosine distance alleviated this issue by emphasizing the orientation of document vectors rather than their magnitude, leading to a substantial performance improvement. Nevertheless, the computational cost of KNN remains high because predictions require comparing each test document with all training samples. As the dataset grows, this approach becomes increasingly inefficient in both memory consumption and inference time.

These findings indicate that KNN is generally less suitable for large-scale sparse text classification tasks than linear or probabilistic models.

---

# 12. Final Conclusion

This study evaluated the performance of K-Nearest Neighbors using multiple neighborhood sizes, feature extraction settings, and distance metrics for binary text classification.

The results show that **distance metric selection has a far greater impact than simply increasing the number of neighbors**. Replacing Euclidean distance with cosine similarity improved classification accuracy by approximately **11 percentage points**, demonstrating the importance of choosing similarity measures that align with the characteristics of sparse textual data.

Despite these improvements, KNN consistently underperformed compared with Logistic Regression and the Naive Bayes classifiers. Its reliance on distance calculations in high-dimensional sparse feature spaces, together with its expensive prediction phase, limits its effectiveness for text mining applications.

The best-performing configuration employed **CountVectorizer (10,000 features, min_df=2, bigrams)** together with **KNN (K=11, cosine distance)**, achieving an accuracy of **75.31%**, an ROC-AUC of **0.7523**, and a cross-validation accuracy of **73.47%**.

Overall, the experimental findings suggest that while KNN can serve as a useful baseline for text classification, it is not the most appropriate algorithm for high-dimensional Bag-of-Words representations. Linear models such as Logistic Regression and probabilistic models such as Multinomial or Bernoulli Naive Bayes remain more accurate, computationally efficient, and scalable for large text classification tasks.
